# Notebook 4/4 — Nested/Scattering-Path Robustness Test

**Evaluation only — no training, no time budget needed. Should finish in minutes.**

Supervisor's idea: does the compressed model lose its ability to recover known paths
*faster* than the teacher when an unexpected interfering path is added?

### Protocol (many independent scenes, not one)
```
for each of N_SCENES independent scenes:
    draw 3 "principal" paths (own random angles + gains, L=3 distribution -- same as
    the standard evaluation protocol elsewhere in this project)

    for each nuisance power level in {-20, -10, 0} dB (relative to total principal power):
        add a 4th "nuisance" path at that power, own random angle
        SAME principal paths + SAME noise realization across the 3 power levels
        within this scene -- only the nuisance power changes (paired comparison,
        isolates the effect of interference strength from scene-to-scene difficulty)

        evaluate teacher AND student on the exact same Y
        ask for L=4 peaks, match all 4 (Hungarian, reusing the ORIGINAL evaluator)
        report principal-path recovery and nuisance-path detection SEPARATELY
```
Two SNR levels tested (0 dB -- hard, 15 dB -- moderate) x 3 nuisance-power levels x N_SCENES.

### What gets measured
```
Pd_principal(power)  -- recovery of the original 3 paths, as interference grows
Pd_nuisance(power)   -- detection of the new path itself
```
compared as teacher vs student. The open question: does the GAP between them widen
as nuisance power increases (compression = fragile to interference) or stay flat
(compression = uniformly weaker, but not specifically fragile to clutter)?

### Caveats respected (per the plan document)
- Principal-path gains use the standard L=3 distribution, NOT renormalized when the
  nuisance is added (so principal amplitudes stay comparable to the main evaluation)
- Realized total power / SNR is reported explicitly, not hidden
- The strongest-gain path is never called "LOS" -- this is an abstract simulation
  with no propagation geometry, so it's just "dominant path"

### Kaggle attach
- `dldoa-source-code` (DL_DOA folder + pretrained teacher weights)
- your new dataset with `student_r8_magnitude_lambda0.5.weights.h5`

In [ ]:
# Cell 1 — Setup
import importlib, subprocess, sys, time
T_START = time.time()

try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)

import os, math, json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, Activation, Add, Conv2DTranspose
from tensorflow.keras.models import Model
from scipy.optimize import linear_sum_assignment

tf.keras.backend.clear_session()
tf.get_logger().setLevel('ERROR')
np.random.seed(42); tf.random.set_seed(42)

gpus = tf.config.list_physical_devices('GPU')
print(f'TF: {tf.__version__}  |  GPU: {gpus}')
for g in gpus: tf.config.experimental.set_memory_growth(g, True)

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)
print('✅ Ready')

In [ ]:
# Cell 2 — Locate repo source, teacher weights, student weights
def find_path(name_pattern):
    from pathlib import Path
    for root in ['/kaggle/input', '/kaggle/working', '.', '/content']:
        if not os.path.isdir(root): continue
        for p in Path(root).rglob(name_pattern):
            if p.is_file(): return str(p)
    return None

SRC_MODEL_FILE = find_path('tvt_models.py')
assert SRC_MODEL_FILE is not None, 'DL_DOA source not found -- attach dldoa-source-code'
DL_DOA_DIR = os.path.dirname(os.path.dirname(SRC_MODEL_FILE))
print(f'DL_DOA dir: {DL_DOA_DIR}')

TEACHER_WEIGHTS_PATH = find_path('inf_model_007_256_resnet.h5')
assert TEACHER_WEIGHTS_PATH is not None, 'pretrained teacher weights not found'
print(f'Teacher weights: {TEACHER_WEIGHTS_PATH}')

STUDENT_WEIGHTS_PATH = find_path('student_r8_magnitude*.weights.h5')
assert STUDENT_WEIGHTS_PATH is not None, 'student weights not found -- attach your uploaded dataset'
print(f'Student weights: {STUDENT_WEIGHTS_PATH}')

In [ ]:
# Cell 3 — Import ORIGINAL evaluator + Resnet (metric-critical, never reimplemented)
sys.path.insert(0, DL_DOA_DIR)
from src.tvt_models import Resnet
from src.TVT_Blob_Inference import get_blob_detector, get_blob_peaks, peaks_to_angles, get_ang_difference, filter_angles
from src.tvt_data_generation_v3 import permute_pairs   # Hungarian matching, same as the rest of the project

print('✅ Imported Resnet + original evaluator functions')

In [ ]:
# Cell 4 — Load teacher (full) and student (pruned r=8, already fine-tuned)
teacher = Resnet(input_shape=(64, 64, 2))
teacher.load_weights(TEACHER_WEIGHTS_PATH)
teacher.trainable = False

def res_conv_pruned(x, r, out_filters=12):
    skip = x
    x = Conv2D(r, 5, padding='same')(x); x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(out_filters, 5, padding='same')(x); x = BatchNormalization()(x)
    x = Add()([x, skip]); x = Activation('relu')(x)
    return x

def build_pruned_resnet(r, n_blocks=64, input_shape=(64, 64, 2), name=None):
    x_in = Input(shape=input_shape)
    x = Conv2DTranspose(12, (5, 5), strides=(2, 2), padding='same')(x_in)
    for _ in range(n_blocks):
        x = res_conv_pruned(x, r)
    x = Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same')(x)
    return Model(x_in, x, name=name or f'PrunedResNet-r{r}')

student = build_pruned_resnet(8, name='Student-r8-magnitude')
student.load_weights(STUDENT_WEIGHTS_PATH)
student.trainable = False

print(f'✅ Teacher: {teacher.count_params():,} params')
print(f'✅ Student: {student.count_params():,} params '
      f'({(1-student.count_params()/teacher.count_params())*100:.1f}% smaller)')

In [ ]:
# Cell 5 — Physics (paper-exact, self-contained -- same functions used throughout this project)
class _H(np.ndarray):
    @property
    def H(self): return self.conj().transpose()

def ev(n, angle):
    return ((1/np.sqrt(n)) * np.exp(-1j*np.pi*np.cos(angle)*np.arange(n))).reshape(-1,1)

def make_F(P, nt):
    phi = np.arccos((1/np.pi)*np.angle(np.exp( 1j*(2*np.pi/P)*np.arange(P))))
    F = np.zeros((nt, P), dtype=complex)
    for i, ph in enumerate(phi): F[:, i] = ev(nt, ph).ravel()
    return F

def make_W(Q, nr):
    phi = np.arccos((1/np.pi)*np.angle(np.exp(-1j*(2*np.pi/Q)*np.arange(Q))))
    W = np.zeros((nr, Q), dtype=complex)
    for i, ph in enumerate(phi): W[:, i] = ev(nr, ph).ravel()
    return W

def gen_channel(nr, nt, phi_l, psi_l, alpha_l):
    Hm = np.zeros((nr, nt), dtype=complex)
    for a, phi, psi in zip(alpha_l, phi_l, psi_l):
        Hm += a * (ev(nr, psi) * ev(nt, phi).view(_H).H)
    return np.sqrt(nt * nr) * Hm

def gen_points(L, delta=np.pi/6, max_try=20000):
    pts = []
    for _ in range(max_try):
        if len(pts) == L: break
        x, y = np.random.uniform(0, np.pi), np.random.uniform(0, np.pi)
        if all(math.hypot(x-p[0], y-p[1]) >= delta for p in pts): pts.append((x, y))
    if len(pts) < L: raise RuntimeError(f'Cannot place {L} points with delta={delta:.3f}')
    return pts

P = Q = nt = nr = 16
F_FIXED = make_F(P, nt)
W_FIXED = make_W(Q, nr)
print('✅ Physics ready (P=Q=nt=nr=16, matching the main evaluation protocol)')

In [ ]:
# Cell 6 — Nested-scene generator: 3 principal paths (own random draw per scene) +
# 1 nuisance path at a swept power level. Principal paths + noise realization are
# IDENTICAL across the power sweep within one scene (paired comparison).

def make_scene(rng):
    """Draw 3 principal paths + 1 nuisance-position (all 4 min-separated), and one
    noise seed, to be reused across every power level tested for this scene."""
    pts = gen_points(4, delta=np.pi/6)   # 3 principal + 1 nuisance position, all separated
    phi_l = np.array([p[0] for p in pts]); psi_l = np.array([p[1] for p in pts])

    # Principal gains: standard L=3 distribution (same as the main evaluation protocol) --
    # NOT renormalized when the nuisance is added, so amplitudes stay comparable.
    L_principal = 3
    alpha_principal = (np.sqrt(1/L_principal)/np.sqrt(2)) * (rng.standard_normal(L_principal) + 1j*rng.standard_normal(L_principal))
    noise_seed = rng.integers(0, 2**31 - 1)
    return phi_l, psi_l, alpha_principal, noise_seed

def make_trial(phi_l, psi_l, alpha_principal, noise_seed, nuisance_power_db, snr_db):
    """phi_l/psi_l: length-4 (3 principal + 1 nuisance, index 3). Returns (data, feat, realized_snr_db)."""
    principal_power = np.sum(np.abs(alpha_principal)**2)
    nuisance_ratio = 10**(nuisance_power_db/10)
    nuisance_mag = np.sqrt(nuisance_ratio * principal_power)
    rng_local = np.random.default_rng(noise_seed + 777)   # deterministic nuisance phase per scene
    nuisance_phase = rng_local.uniform(0, 2*np.pi)
    alpha_nuisance = nuisance_mag * np.exp(1j*nuisance_phase)
    alpha_all = np.concatenate([alpha_principal, [alpha_nuisance]])   # length 4

    Hm = gen_channel(nr, nt, phi_l, psi_l, alpha_all)

    rng_noise = np.random.default_rng(noise_seed)   # SAME noise realization across power levels
    var = 10**(-snr_db/10); s = np.sqrt(var/2)
    Z = s * (rng_noise.standard_normal((Q, P)) + 1j*rng_noise.standard_normal((Q, P)))

    Y = (W_FIXED.view(_H).H @ Hm) @ F_FIXED + Z
    data = np.stack([Y.real, Y.imag], axis=-1).astype(np.float32)   # (16,16,2) native, matches ChannelEst notebook style
    # upsample to 64x64x2 to match the ResNet's expected input (zoom=4 for P=16)
    import scipy.ndimage
    data_up = np.stack([
        scipy.ndimage.zoom(data[:,:,0], 4, order=0),
        scipy.ndimage.zoom(data[:,:,1], 4, order=0),
    ], axis=-1).astype(np.float32)

    total_signal_power = np.sum(np.abs(alpha_all)**2)
    realized_snr_db = 10*np.log10(total_signal_power / var)

    feat = np.stack([psi_l, phi_l]).astype(np.float32)   # (2,4) -- index 0-2 principal, 3 nuisance
    return data_up, feat, realized_snr_db

print('✅ Scene/trial generator ready')

In [ ]:
# Cell 7 — Evaluation: split principal-path recovery from nuisance-path detection,
# reusing the ORIGINAL evaluator's matching/metric functions (component-wise Pd,
# same convention as every other notebook in this project).

DETECTOR = get_blob_detector()

def prepare_for_metric_L4(angles_est, feat):
    """Same logic as TVT_Blob_Inference.prepare_for_metric, but keeping the Hungarian-
    matched order explicit so principal (idx 0-2) vs nuisance (idx 3) can be split."""
    L = feat.shape[-1]   # = 4
    if len(angles_est[0]) < L:
        return np.full((2, L), np.nan), np.full((2, L), np.nan)
    angles_est = (angles_est[0][:L], angles_est[1][:L])
    pairs_est = list(zip(angles_est[0], angles_est[1]))
    pairs_true = list(zip(feat[0], feat[1]))
    permuted_pairs = permute_pairs(pairs_true, pairs_est)   # Hungarian; row order == pairs_true order
    first_pairs = [p[0] for p in permuted_pairs]; second_pairs = [p[1] for p in permuted_pairs]
    gt_angles = np.array(list(zip(*first_pairs)))
    pred_angles = np.array(list(zip(*second_pairs)))
    return gt_angles, pred_angles

def evaluate_trial(model, data, feat, max_deg_error=1.0):
    pred = model(np.expand_dims(data, 0), training=False)[0]
    peaks, amps = get_blob_peaks(pred, DETECTOR)
    order = np.argsort(-amps); peaks = peaks[order[:4]]   # L=4 known -- ask for all 4
    angles_est = peaks_to_angles(peaks, sigma=0.07, grid_size=256)
    gt_angles, pred_angles = prepare_for_metric_L4(angles_est, feat)

    if np.isnan(pred_angles).any():
        return None   # sample dropped entirely, matching the original evaluator's convention

    diffs_principal = get_ang_difference(gt_angles[:, :3], pred_angles[:, :3])   # 6 values (3 psi + 3 phi)
    diffs_nuisance  = get_ang_difference(gt_angles[:, 3:4], pred_angles[:, 3:4])  # 2 values
    good_p, bad_p = filter_angles(diffs_principal, max_deg_error)
    good_n, bad_n = filter_angles(diffs_nuisance, max_deg_error)
    return len(good_p), len(good_p)+len(bad_p), len(good_n), len(good_n)+len(bad_n)

print('✅ Evaluation functions ready')

In [ ]:
# Cell 8 — Main experiment: N_SCENES independent scenes x 2 SNR levels x 3 nuisance powers
N_SCENES = 200          # independent scenes -- increase if time permits, decrease if too slow
SNR_LEVELS = [0, 15]    # dB -- one hard, one moderate condition
NUISANCE_POWERS_DB = [-20, -10, 0]   # relative to total principal power

t0 = time.time()
results = {}   # results[(snr, power)][model_name] = dict(det_p, tot_p, det_n, tot_n, n_dropped)

for snr in SNR_LEVELS:
    for power in NUISANCE_POWERS_DB:
        acc = {name: dict(det_p=0, tot_p=0, det_n=0, tot_n=0, n_dropped=0) for name in ['teacher', 'student']}
        rng = np.random.default_rng(1000 + snr)   # SAME scene sequence across power levels, per SNR
        for scene_i in range(N_SCENES):
            phi_l, psi_l, alpha_principal, noise_seed = make_scene(rng)
            data, feat, realized_snr = make_trial(phi_l, psi_l, alpha_principal, noise_seed, power, snr)
            for name, model in [('teacher', teacher), ('student', student)]:
                r = evaluate_trial(model, data, feat)
                if r is None:
                    acc[name]['n_dropped'] += 1
                else:
                    dp, tp, dn, tn = r
                    acc[name]['det_p'] += dp; acc[name]['tot_p'] += tp
                    acc[name]['det_n'] += dn; acc[name]['tot_n'] += tn
        results[(snr, power)] = acc
        print(f'SNR={snr:>3}dB  nuisance={power:>4}dB  '
              f'teacher Pd_principal={acc["teacher"]["det_p"]/max(acc["teacher"]["tot_p"],1):.4f}  '
              f'student Pd_principal={acc["student"]["det_p"]/max(acc["student"]["tot_p"],1):.4f}  '
              f'({time.time()-t0:.0f}s elapsed)')

print(f'\n✅ Experiment complete in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 9 — Results table + plots: does the teacher-student GAP widen as interference grows?
def pd_of(acc, key_num, key_den):
    return acc[key_num] / acc[key_den] if acc[key_den] > 0 else float('nan')

print('='*95)
print(f'{"SNR":>5} {"Nuisance":>10} | {"Teach Pd_p":>11} {"Stud Pd_p":>10} {"ΔPd_p":>8} | '
      f'{"Teach Pd_n":>11} {"Stud Pd_n":>10} {"ΔPd_n":>8}')
print('-'*95)
table_rows = []
for snr in SNR_LEVELS:
    for power in NUISANCE_POWERS_DB:
        acc = results[(snr, power)]
        t_pdp = pd_of(acc['teacher'], 'det_p', 'tot_p'); s_pdp = pd_of(acc['student'], 'det_p', 'tot_p')
        t_pdn = pd_of(acc['teacher'], 'det_n', 'tot_n'); s_pdn = pd_of(acc['student'], 'det_n', 'tot_n')
        d_p = s_pdp - t_pdp; d_n = s_pdn - t_pdn
        table_rows.append((snr, power, t_pdp, s_pdp, d_p, t_pdn, s_pdn, d_n))
        print(f'{snr:>5} {power:>9}dB | {t_pdp:>11.4f} {s_pdp:>10.4f} {d_p:>+8.4f} | '
              f'{t_pdn:>11.4f} {s_pdn:>10.4f} {d_n:>+8.4f}')
print('='*95)

fig, axs = plt.subplots(1, len(SNR_LEVELS), figsize=(6*len(SNR_LEVELS), 4.5), sharey=True)
if len(SNR_LEVELS) == 1: axs = [axs]
for ax, snr in zip(axs, SNR_LEVELS):
    t_vals = [pd_of(results[(snr,p)]['teacher'], 'det_p', 'tot_p') for p in NUISANCE_POWERS_DB]
    s_vals = [pd_of(results[(snr,p)]['student'], 'det_p', 'tot_p') for p in NUISANCE_POWERS_DB]
    ax.plot(NUISANCE_POWERS_DB, t_vals, 's--', color='steelblue', label='Teacher')
    ax.plot(NUISANCE_POWERS_DB, s_vals, 'o-', color='crimson', label='Student (r8, magnitude)')
    ax.set_xlabel('Nuisance power (dB rel. to principal)'); ax.set_title(f'SNR = {snr} dB')
    ax.legend(); ax.grid(alpha=.3)
axs[0].set_ylabel('Pd (principal paths)')
plt.suptitle('Does compression make principal-path recovery MORE fragile to interference?', y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'nested_path_robustness.png'), dpi=140, bbox_inches='tight')
plt.show()

print()
for snr in SNR_LEVELS:
    gaps = [s_pdp - t_pdp for (s,p,t_pdp,s_pdp,dp,tn,sn,dn) in table_rows if s == snr
            for t_pdp, s_pdp in [(t_pdp, s_pdp)]]
    d_low = table_rows[[r[0]==snr and r[1]==NUISANCE_POWERS_DB[0] for r in table_rows].index(True)][4]
    d_high = table_rows[[r[0]==snr and r[1]==NUISANCE_POWERS_DB[-1] for r in table_rows].index(True)][4]
    widening = d_high - d_low
    print(f'SNR={snr}dB: gap at weakest nuisance ({NUISANCE_POWERS_DB[0]}dB) = {d_low:+.4f}, '
          f'at strongest ({NUISANCE_POWERS_DB[-1]}dB) = {d_high:+.4f}  '
          f'-> {"WIDENS" if widening < -0.01 else ("NARROWS" if widening > 0.01 else "~flat")} '
          f'by {widening:+.4f}')

In [ ]:
# Cell 10 — Save results
results_serializable = {
    f'snr{snr}_power{power}': {
        name: {k: int(v) for k, v in acc.items()} for name, acc in results[(snr,power)].items()
    } for snr in SNR_LEVELS for power in NUISANCE_POWERS_DB
}
out = {
    'n_scenes': N_SCENES, 'snr_levels': SNR_LEVELS, 'nuisance_powers_db': NUISANCE_POWERS_DB,
    'student_params': int(student.count_params()), 'teacher_params': int(teacher.count_params()),
    'results': results_serializable,
    'table_rows': [[float(x) for x in row] for row in table_rows],
}
with open(os.path.join(OUT_DIR, 'nested_path_results.json'), 'w') as f:
    json.dump(out, f, indent=2)
print(f'Saved: {os.path.join(OUT_DIR, "nested_path_results.json")}')
print()
print('⚠️ Remember: Save Version -> Save & Run All (Commit) so this persists.')